In [2]:
"""
SQL Assignment - Olympics & Sales Data Analysis
================================================
This notebook uses sqlite3 and pandas to query datasets
loaded into an in-memory SQLite database.

Datasets:
- athletes_table : Olympic athlete events (1896-2016)
- regions_table  : NOC country codes and region names
- sales_table    : Chipotle order data
"""

import sqlite3
import pandas as pd

In [6]:
# Load datasets
athletes = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/athlete_events.csv", low_memory=False)
regions = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/noc_regions.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv", sep='\t')


# Connect to SQLite in-memory DB
conn = sqlite3.connect(":memory:")

# Write DataFrames to SQL tables
athletes.to_sql("athletes_table", conn, index=False, if_exists="replace")
regions.to_sql("regions_table", conn, index=False, if_exists="replace")
sales.to_sql("sales_table", conn, index=False, if_exists="replace")

4622

In [ ]:
#Section 1

In [13]:
pd.read_sql_query('SELECT * FROM athletes_table;',conn)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271111,135569,Andrzej ya,M,29.0,179.0,89.0,Poland-1,POL,1976 Winter,1976,Winter,Innsbruck,Luge,Luge Mixed (Men)'s Doubles,NaN
271112,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Individual",NaN
271113,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Team",NaN
271114,135571,Tomasz Ireneusz ya,M,30.0,185.0,96.0,Poland,POL,1998 Winter,1998,Winter,Nagano,Bobsleigh,Bobsleigh Men's Four,NaN


In [17]:
#1.Count the total number of medals won by each country and display the top 5.
pd.read_sql_query("""
    SELECT NOC AS Country, 
           COUNT(Medal) AS Total_Medals
    FROM athletes_table
    WHERE Medal IS NOT NULL
    GROUP BY NOC
    ORDER BY Total_Medals DESC
    LIMIT 5;
""", conn)

,Country,Total_Medals
0,USA,5637
1,URS,2503
2,GER,2165
3,GBR,2068
4,FRA,1777


In [23]:
#2.Calculate the average age of athletes who won a Gold medal.
pd.read_sql_query(""" 
SELECT AVG(age) AS won_gold
FROM athletes_table
WHERE Medal='Gold';
""", conn)

,won_gold
0,25.901013


In [27]:
#3.How many distinct events are there in each sport?
pd.read_sql_query(""" 
SELECT Sport,COUNT(DISTINCT Event) as event_no
FROM athletes_table
GROUP BY Sport;
""", conn)

,Sport,event_no
0,Aeronautics,1
1,Alpine Skiing,10
2,Alpinism,1
3,Archery,29
4,Art Competitions,29
...,...,...
61,Tug-Of-War,1
62,Volleyball,2
63,Water Polo,2
64,Weightlifting,21


In [29]:
#4.Show all athletes from the United States (NOC = 'USA').
pd.read_sql_query(""" 
SELECT DISTINCT Name
FROM athletes_table
WHERE NOC='USA';
""", conn)

,Name
0,Per Knut Aaland
1,John Aalberg
2,Stephen Anthony Abas
3,"David ""Dave"" Abbott"
4,Jeremy Abbott
...,...
9647,Frank Thomas Zuna
9648,David Santos Zuniga
9649,Rami Zur
9650,"Victor Andrew ""Vic"" Zwolak"


In [31]:
#5.Count how many medals were awarded each year.
pd.read_sql_query(""" 
SELECT COUNT(Medal) AS medal_awarded, year
FROM athletes_table
WHERE MEDAL IS NOT NULL
GROUP BY year;
""", conn)

,medal_awarded,Year
0,143,1896
1,604,1900
2,486,1904
3,458,1906
4,831,1908
5,941,1912
6,1308,1920
7,962,1924
8,823,1928
9,739,1932


In [34]:
#6.Find all athlete records where height or weight is missing.
pd.read_sql_query(""" 
SELECT Name,Height, Weight
FROM athletes_table
WHERE height IS NULL OR weight IS NULL;
""", conn)

,Name,Height,Weight
0,Gunnar Nielsen Aaby,NaN,NaN
1,Edgar Lindenau Aabye,NaN,NaN
2,"Cornelia ""Cor"" Aalten (-Strannood)",168.0,NaN
3,"Cornelia ""Cor"" Aalten (-Strannood)",168.0,NaN
4,"Einar Ferdinand ""Einari"" Aalto",NaN,NaN
...,...,...,...
64258,Marius Edmund Zwiller,NaN,NaN
64259,Werner Zwingli,NaN,NaN
64260,Werner Zwingli,NaN,NaN
64261,Jan (Johann-) Zybert (Siebert-),NaN,NaN


In [43]:
#7.Replace the missing height with the average athlete height.
conn.execute(""" 
UPDATE athletes_table
SET Height=(
  SELECT AVG(Height) AS avg_height
FROM athletes_table
WHERE height IS NOT NULL)             
WHERE Height IS NULL;
""")
conn.commit()

In [44]:
pd.read_sql_query('SELECT * FROM sales_table;',conn)

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98
...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75


In [49]:
#8.Return total sales value per item.
pd.read_sql_query(""" 
SELECT SUM(quantity * CAST(REPLACE(item_price, '$', '') AS FLOAT)) AS total_sales, item_name
FROM sales_table
GROUP BY item_name;
""", conn)

,total_sales,item_name
0,369.93,6 Pack Soft Drink
1,672.36,Barbacoa Bowl
2,894.75,Barbacoa Burrito
3,138.71,Barbacoa Crispy Tacos
4,106.40,Barbacoa Salad Bowl
5,250.46,Barbacoa Soft Tacos
6,649.18,Bottled Water
7,74.00,Bowl
8,44.40,Burrito
9,191.84,Canned Soda


In [52]:
#9.Show the top 5 records with the highest item_price.
pd.read_sql_query(""" 
SELECT item_name, CAST(REPLACE(item_price, '$', '') AS FLOAT) AS item_price
FROM sales_table
ORDER BY item_price DESC
LIMIT 5;
""", conn)

,item_name,item_price
0,Chips and Fresh Tomato Salsa,44.25
1,Carnitas Bowl,35.25
2,Chicken Burrito,35.00
3,Chicken Burrito,35.00
4,Veggie Burrito,33.75


In [55]:
#10.How many unique customer orders are there? (Assume each order_id is a customer.)
pd.read_sql_query(""" 
SELECT COUNT(DISTINCT order_id) AS customer_orders
FROM sales_table
;
""", conn)

,customer_orders
0,1834


In [ ]:
#This is now section 2 

In [ ]:
#Write one query for this section.

#Find countries with high-performing athletes in the Olympics. Use at least JOIN, NESTED QUERY, CASE, and optionally WITH.

#For each country:

#Count the number of athletes who won at least one medal.
#Determine the average age of those medalists.
#Create a new column called performance:
#'High' if average age is below 25
#'Medium' if between 25 and 30
#'Low' if above 30

In [59]:
pd.read_sql_query(""" 
WITH medalists AS(
SELECT DISTINCT a.Name, a.NOC, a.Age
FROM athletes_table a
WHERE a.Name IN (
    SELECT Name
    FROM athletes_table  
    WHERE Medal IS NOT NULL )
                  ),
country_stats AS (
       SELECT r.region AS Country,
           COUNT(DISTINCT m.Name) AS Total_Medalists,
           ROUND(AVG(m.Age), 1) AS Average_Age
    FROM medalists m
    JOIN regions_table r ON m.NOC = r.NOC
    GROUP BY r.region
                             ) 
SELECT Country,
       Total_Medalists,
       Average_Age,
       CASE
           WHEN Average_Age < 25 THEN 'High'
           WHEN Average_Age BETWEEN 25 AND 30 THEN 'Medium'
           ELSE 'Low'
       END AS Performance
FROM country_stats
ORDER BY Total_Medalists DESC;
""",conn)

,Country,Total_Medalists,Average_Age,Performance
0,USA,3851,25.6,Medium
1,Russia,2625,25.7,Medium
2,Germany,2572,25.9,Medium
3,UK,1608,28.1,Medium
4,France,1280,27.6,Medium
...,...,...,...,...
134,Swaziland,1,30.0,Medium
135,Togo,1,26.7,Medium
136,Tonga,1,26.0,Medium
137,Turkmenistan,1,20.0,High
